## Access this Notebook

You can launch this notebook in the US GHG Center JupyterHub by clicking the link below. If you are a new user, you should first sign up for the hub by filling out this [**request form**](https://docs.google.com/forms/d/e/1FAIpQLSdJEOVa3rEjRKl2o0kqCCYM8io2cT8FplnYVnfqR3WV9IENtg/viewform) and providing the required information.

Access the [**Global Mangrove Distribution, Aboveground Biomass, and Canopy Height**](https://hub.ghg.center/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FUS-GHG-Center%2Fghgc-docs&urlpath=lab%2Ftree%2Fghgc-docs%2Fuser_data_notebooks%2Fcms-global-map-mangrove_User_Notebook.ipynb&branch=main) notebook in the US GHG Center JupyterHub (requires access).

## Table of Contents
- [Data Summary and Application](#data-summary-and-application)
- [Approach](#approach)
- [About the Data](#about-the-data)
- [Install the Required Libraries](#install-the-required-libraries)
- [Query the STAC API](#query-the-stac-api)
- [Explore Available Regions](#explore-available-regions)
- [Visualize Mangrove Data on a Map](#visualize-mangrove-data-on-a-map)
- [Calculate Zonal Statistics](#calculate-zonal-statistics)
- [Explore Additional Variables and Regions](#explore-additional-variables-and-regions)
- [Summary](#summary)

## Data Summary and Application
- **Spatial coverage**: Global mangrove extent
- **Spatial resolution**: 30 meters  
- **Temporal extent**: Nominal year 2000
- **Temporal resolution**: Single time point
- **Unit**: 
  - Aboveground biomass (agb): Megagrams per hectare (Mg/ha)
  - Maximum height 95th percentile (hmax95): meters (m)
  - Basal area height 95th percentile (hba): meters (m)
- **Utility**: Blue carbon accounting, coastal ecosystem management, climate change mitigation

For more information, visit the [Global Mangrove Distribution, Aboveground Biomass, and Canopy Height](https://earth.gov/ghgcenter/data-catalog/cms-mangrove-agb-canopyheight-grid-v1_3) data overview page.

## Approach

1. Identify and explore the Global Mangrove Distribution, Aboveground Biomass, and Canopy Height dataset using the GHGC API `/stac` endpoint
2. Examine available regions and assets (biomass, height metrics) in the collection
3. Visualize mangrove data for a specific region (Australia) on an interactive map
4. Define an area of interest and calculate zonal statistics
5. Create interactive widgets to explore different variables and regions
6. Display mangrove characteristics for other regions globally

## About the Data

This dataset characterizes the global distribution, biomass, and canopy height of mangrove-forested wetlands based on remotely sensed and in situ field measurement data. Estimates of (1) mangrove aboveground biomass (AGB), (2) maximum canopy height (height of the tallest tree) [hmax95], and (3) basal-area weighted height (individual tree heights weighted in proportion to their basal area) [hba] for the nominal year 2000 were derived across a 30-meter resolution global mangrove ecotype extent map using remotely-sensed canopy height measurements and region-specific allometric models.

Mangroves are unique coastal forests found in tropical and subtropical regions that:
- Store substantial amounts of carbon in their biomass and soils
- Protect coastlines from erosion and storm damage
- Support diverse marine and terrestrial species
- Provide livelihoods for coastal communities

For more information regarding this dataset, please visit the [Global Mangrove Distribution, Aboveground Biomass, and Canopy Height](https://earth.gov/ghgcenter/data-catalog/cms-mangrove-agb-canopyheight-grid-v1_3) data overview page.

## Terminology

Navigating data via the GHGC API, you will encounter terminology that is different from browsing in a typical filesystem. We'll define some terms here which are used throughout this notebook.

-  `catalog`:    All datasets available at the `/stac` endpoint
-  `collection`: A specific dataset, e.g. Global Mangrove Distribution, Aboveground Biomass, and Canopy Height
-  `item`:       One granule in the dataset, e.g. mangrove data for a specific country/region
-  `asset`:      A variable available within the granule, e.g. mangrove-agb, mangrove-hmax95, or mangrove-hba95
-  `STAC API`:   **Sp**atio**T**emporal **A**sset **C**atalogs - Endpoint for fetching metadata about available datasets
-  `Raster API`: Endpoint for fetching data itself, for imagery and statistics

## Install the Required Libraries

Required libraries are pre-installed on the GHG Center Hub. If you need to run this notebook elsewhere, please install them with this line in a code cell:

%pip install requests folium rasterstats pystac_client pandas matplotlib ipywidgets --quiet

In [1]:
# Import required libraries
import os
import json
import requests
import numpy as np
import pandas as pd
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# For making maps
import folium
import folium.plugins
from folium import Map, TileLayer

# For working with geospatial data
import geopandas as gpd
from shapely.geometry import box

# For talking to the STAC API
from pystac_client import Client

# Import GHG Center utilities
import ghgc_utils

print("All libraries imported successfully!")

All libraries imported successfully!


## Query the STAC API

### **STAC API Collection Names**

Now, you must fetch the dataset from the [**STAC API**](https://earth.gov/ghgcenter/api/stac/) by defining its associated STAC API collection ID as a variable. 
The collection ID, also known as the **collection name**, for the Global Mangrove Distribution, Aboveground Biomass, and Canopy Height dataset is [**cms-mangrove-agb-canopyheight-grid-v1.3**](https://earth.gov/ghgcenter/api/stac/collections/cms-mangrove-agb-canopyheight-grid-v1.3).*

**You can find the collection name of any dataset on the GHGC data portal by navigating to the dataset landing page within the data catalog. The collection name is the last portion of the dataset landing page's URL, and is also listed in the pop-up box after clicking "ACCESS DATA."*

In [2]:
# Provide the STAC and RASTER API endpoints
# The STAC API is a catalog of all the existing data collections that are stored in the GHG Center.
STAC_API_URL = "https://earth.gov/ghgcenter/api/stac"

# The RASTER API is used to fetch collections for visualization
RASTER_API_URL = "https://earth.gov/ghgcenter/api/raster"

# The collection name is used to fetch the dataset from the STAC API. First, we define the collection name as a variable
collection_name = "cms-mangrove-agb-canopyheight-grid-v1.3"

In [3]:
# Fetch the collection from the STAC API using the appropriate endpoint
# The 'pystac_client' library makes a HTTP request
catalog = Client.open(STAC_API_URL)
collection = catalog.get_collection(collection_name)

# Print the properties of the collection to the console
collection

<CollectionClient id=cms-mangrove-agb-canopyheight-grid-v1.3>

Examining the contents of our `collection` under the `extent/spatial/temporal` variable, we note that data is only available for the nominal year 2000.

In [4]:
search = catalog.search(
    collections=collection_name,
    datetime=['2000-01-01T00:00:00Z']
)

# Find all items in collection
items = search.item_collection()

# Take a look at the number of items we found
print(f"# items in date range: {len(items)}")

#Create a list of unique regions to subset for later use. 
#Also print the items in the collection
unique_regions = []
for item in items:
    print(item)
    unique_regions.append(item.id.split('-')[-1])

unique_regions = np.unique(unique_regions)
print(f'Number of unique regions is {len(unique_regions)}')

# items in date range: 121
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Yemen>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-WallisAndFutuna>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-VirginIslandsUs>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Vietnam>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Venezuela>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Vanuatu>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-UnitedStates>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-UnitedArabEmirates>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Tuvalu>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-TurksAndCaicosIslands>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-TrinidadAndTobago>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Tonga>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Togo>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-TimorLeste>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-Thailand>
<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-

In [5]:
# Before we go further, let's pick which asset to focus on for the first set of plots within the notebook. 
# This dataset has three assets to choose from:
asset_name = "agb"

## Visualize Mangrove Data on a Map
You will now explore mangrove distribution and characteristics for different regions and visualize the results on a map using `folium`.

### Fetch Imagery from Raster API
Here we get information from the `Raster API` which we will add to our map in the next section.

In [6]:
# Generate the plotting legend unit information
units = {'agb':'Mg ha-1','hmax95':'meters','hba':'meters'}

In [7]:
# Use the information from the dropdown box above to grab the proper collection item
observation_1 = [i for i in items if 'Australia' in i.id][0]

# Extract collection name and item ID
collection_id = observation_1.collection_id
item_id = observation_1.id

In [8]:
object = observation_1.assets[asset_name]
raster_bands = object.extra_fields.get("raster:bands", [{}])
rescale_values = {
    "max": raster_bands[0].get("histogram", {}).get("max"),
    "min": 0,
}

raster_bands

[{'scale': 1.0,
  'nodata': -9999.0,
  'offset': 0.0,
  'sampling': 'area',
  'data_type': 'float32',
  'histogram': {'max': 145.98110961914062,
   'min': -8.919515609741211,
   'count': 11,
   'buckets': [739881, 1690, 500, 175, 84, 34, 17, 11, 6, 2]},
  'statistics': {'mean': 0.08236342857623923,
   'stddev': 1.6322306092733863,
   'maximum': 145.98110961914062,
   'minimum': -8.919515609741211,
   'valid_percent': 100.0}}]

Now, you will pass the `item id`, `collection name`, `asset name`, and the `rescale values` to the Raster API endpoint, along with a colormap. This step tells the Raster API which collection, item, and asset you want to view, specifying the colormap and colorbar ranges to use for visualization. The API returns a JSON with information about the requested image. Each image will be referred to as a tile.

In [9]:
# Choose a colormap for displaying the tiles
# Make sure that the capitalization matches Matplotlib standards
# For more information on Colormaps in Matplotlib, please visit https://matplotlib.org/stable/users/explain/colors/colormaps.html
color_map = "YlGn"

In [10]:
# Make a GET request to retrieve information for the date specified
observation_1_tile = requests.get(
    f"{RASTER_API_URL}/collections/{collection_id}/items/{item_id}/tilejson.json?"
    f"&assets={asset_name}"
    f"&color_formula=gamma+r+1.05&colormap_name={color_map.lower()}"
    f"&rescale=0,{rescale_values['max']}"
).json()

# Print the properties of the retrieved granule to the console
observation_1_tile

{'tilejson': '2.2.0',
 'version': '1.0.0',
 'scheme': 'xyz',
 'tiles': ['https://earth.gov/ghgcenter/api/raster/collections/cms-mangrove-agb-canopyheight-grid-v1.3/items/cms-mangrove-agb-canopyheight-grid-v1.3-Australia/tiles/WebMercatorQuad/{z}/{x}/{y}@1x?assets=agb&color_formula=gamma+r+1.05&colormap_name=ylgn&rescale=0%2C145.98110961914062'],
 'minzoom': 0,
 'maxzoom': 24,
 'bounds': [112.99986111111112,
  -39.00013888891209,
  154.0001388889217,
  -9.99986111111111],
 'center': [133.5000000000164, -24.5000000000116, 0]}

In [11]:
''' PLOTTING NOTES
The mangrove data is tiled where each country has its own tile and the `center` coordinates are within the middle of the country tile area.
When plotting high-resolution mangrove forests (which can be on the coastlines of large countries such as Australia), plotting the center of the tile does not allow for easy visualization.

To alleviate this, we will manually change the location of the latitude and longitude values when plotting to better assist with visualization.

Location: Bathurst Island
latitude = -11.5
longitude = 130.8

'''

latitude = -11.5
longitude = 130.8

# Note that we specify "tiles=None" because in the next step we're going to set a custom tile to serve as our underlying world map.
map_ = folium.Map(location=(latitude, longitude), zoom_start=10, tiles=None, tooltip = 'test tool tip')
# Specify a custom imagery source for the underlying map
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}.png', name='ESRI World Imagery', attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',overlay='True').add_to(map_)
# Add place labels on top
folium.TileLayer(tiles='https://server.arcgisonline.com/arcgis/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}.png',name='ESRI World Boundaries and Places',attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',overlay='True').add_to(map_)

# Use the 'TileLayer' library display the raster layer
map_layer = TileLayer(
    tiles=observation_1_tile["tiles"][0], # Path to retrieve the tile
    name=f'{items[0].assets[asset_name].title}', # Give this layer a title
    overlay='True', # The layer can be overlaid on the map
    attr="GHG", # Set the attribution
    opacity=0.4, # Adjust the transparency of the layer
)
map_layer.add_to(map_)

# Adjust map elements 
folium.LayerControl(collapsed=False, position='topright').add_to(map_)

# Add colorbar
# We can use one of 'generate_html_colorbar' from the 'ghgc_utils' module 
# to create an HTML colorbar representation.
legend_html = ghgc_utils.generate_html_colorbar(
                color_map,
                rescale_values,
                label=f'{asset_name} ({units.get(asset_name)})'
    )

# Add colorbar to the map
map_.get_root().html.add_child(folium.Element(legend_html))


# Visualizing the map
map_

To perform zonal statistics, first we need to create a polygon. In this use case we are creating a polygon over Bathurst Island, Australia.

**Note** Selecting too large of an AOI will result in an error and no results will be returned. This is due to the large size of the files at a high resolution of 30 meters.

In [12]:
# Give the AOI a name to use in plots later on
aoi_name = "Bathurst Island, Northern Australia"
# This AOI is defined as a GEOJSON.
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "coordinates": [
          [
            # [longitude, latitude]
            [130.0, -12.0],  # Southwest Bounding Coordinate
            [130.0, -11.16], # Southeast Bounding Coordinate
            [131.0, -11.16],   # Northeast Bounding Coordinate
            [131.0, -12.0],    # Northwest Bounding Coordinate
            [130.0, -12.0]   # Closing the polygon at the Southwest Bounding Coordinate
          ]
        ],
        "type": "Polygon"
      }
    }
  ]
}


print("Updated AOI for Bathurst Island, Australia:")
print(aoi)

Updated AOI for Bathurst Island, Australia:
{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {}, 'geometry': {'coordinates': [[[130.0, -12.0], [130.0, -11.16], [131.0, -11.16], [131.0, -12.0], [130.0, -12.0]]], 'type': 'Polygon'}}]}


In [30]:
# Quick Folium map to visualize this AOI
map_ = folium.Map(location=(-11.5, 130.8), zoom_start=9)

# Add AOI to map
folium.GeoJson(aoi, name=aoi_name).add_to(map_)
# Add data layer to map to visualize how many grid cells lie within our AOI
map_layer.add_to(map_)
# Add colorbar
legend_html = ghgc_utils.generate_html_colorbar(color_map,rescale_values,label=f'{items[0].assets[asset_name].title} (ppm)')
map_.get_root().html.add_child(folium.Element(legend_html))
map_

In [13]:
%%time
# The mangrove dataset uses 'datetime' instead of 'start_datetime'
# We need to add the datetime property to make it compatible with ghgc_utils
observation_1_with_datetime = observation_1
if not hasattr(observation_1_with_datetime.properties, 'start_datetime'):
    # Add start_datetime if it doesn't exist
    observation_1_with_datetime.properties['start_datetime'] = observation_1.properties.get('datetime', '2000-01-01T00:00:00Z')

# Statistics will be returned as a Pandas DataFrame
df = ghgc_utils.generate_stats([observation_1_with_datetime],aoi,url=RASTER_API_URL,asset=asset_name, nodata=0)

# Print the first five rows of statistics from our DataFrame
df.head(1)

Generating stats...
Done!
CPU times: user 6.03 ms, sys: 2.27 ms, total: 8.3 ms
Wall time: 16.3 s


,datetime,min,max,mean,count,sum,std,median,majority,minority,unique,histogram,valid_percent,masked_pixels,valid_pixels,percentile_2,percentile_98,date
0,2000-01-01T00:00:00Z,0.00000000000000000000,212.61692810058593750000,6.88728094100952148438,10886400.00000000000000000000,74977696.00000000000000000000,29.01314062802513049633,0.00000000000000000000,0.00000000000000000000,57.41016006469726562500,17.00000000000000000000,"[[10274941, 0, 6392, 48493, 140535, 169842, 12...",100.00000000000000000000,0.00000000000000000000,10893025.00000000000000000000,0.00000000000000000000,128.46847534179687500000,2000-01-01 00:00:00+00:00


## Explore Additional Variables and Regions

Now we will create interactive widgets to explore different mangrove variables (biomass, height metrics) across different regions of the world. This allows for comparative analysis of mangrove characteristics globally.

In [14]:
# Formatting settings for drop-down menus
style = {'description_width':'140px'}
layout = widgets.Layout(width='300px', height='75px')

# Create dropdown widgets
variable = widgets.Dropdown(
    options=['agb', 'hmax95', 'hba'],
    value = 'hmax95', 
    description='Variable:', 
    style=style, 
    layout=layout
)

region_select = widgets.Dropdown(
    options=unique_regions,
    value = 'CostaRica', 
    description='Region:', 
    style=style, 
    layout=layout
)


# Create horizontal box to arrange dropdowns side by side
dropdown_box = widgets.HBox([variable, region_select])

# Display drop-down menus
print('If you change menu selections (e.g., to run another search), do NOT re-run this block!')
print('Re-running will re-set all menus to their defaults!')
display(dropdown_box)

If you change menu selections (e.g., to run another search), do NOT re-run this block!
Re-running will re-set all menus to their defaults!


In [15]:
# Use the information from the dropdown box above to grab the proper collection item
observation_2 = [i for i in items if region_select.value in i.id][0]
print(observation_2)

# Extract collection name and item ID
collection_id = observation_2.collection_id
item_id = observation_2.id


<Item id=cms-mangrove-agb-canopyheight-grid-v1.3-CostaRica>


In [16]:
object = observation_2.assets[variable.value]
raster_bands = object.extra_fields.get("raster:bands", [{}])
rescale_values = {
    "max": raster_bands[0].get("histogram", {}).get("max"),
    "min": 0,
}

print(rescale_values)
print(raster_bands)

{'max': 48.81807327270508, 'min': 0}
[{'scale': 1.0, 'nodata': -9999.0, 'offset': 0.0, 'sampling': 'area', 'data_type': 'float32', 'histogram': {'max': 48.81807327270508, 'min': -2.2742505073547363, 'count': 11, 'buckets': [993811, 862, 705, 637, 495, 347, 228, 149, 97, 45]}, 'statistics': {'mean': 0.059796311772089966, 'stddev': 1.1689403043113449, 'maximum': 48.81807327270508, 'minimum': -2.2742505073547363, 'valid_percent': 100.0}}]


In [17]:
# Choose a colormap for displaying the tiles
# Make sure that the capitalization matches Matplotlib standards
# For more information on Colormaps in Matplotlib, please visit https://matplotlib.org/stable/users/explain/colors/colormaps.html
color_map = "YlGn"

In [18]:
# Make a GET request to retrieve information for the date specified
observation_2_tile = requests.get(
    f"{RASTER_API_URL}/collections/{collection_id}/items/{item_id}/tilejson.json?"
    f"&assets={variable.value}"
    f"&color_formula=gamma+r+1.05&colormap_name={color_map.lower()}"
    f"&rescale=0,{rescale_values['max']}"
).json()

# Print the properties of the retrieved granule to the console
observation_2_tile


{'tilejson': '2.2.0',
 'version': '1.0.0',
 'scheme': 'xyz',
 'tiles': ['https://earth.gov/ghgcenter/api/raster/collections/cms-mangrove-agb-canopyheight-grid-v1.3/items/cms-mangrove-agb-canopyheight-grid-v1.3-CostaRica/tiles/WebMercatorQuad/{z}/{x}/{y}@1x?assets=hmax95&color_formula=gamma+r+1.05&colormap_name=ylgn&rescale=0%2C48.81807327270508'],
 'minzoom': 0,
 'maxzoom': 24,
 'bounds': [-86.09476906457506,
  7.801602899887083,
  -82.5033801756833,
  11.215214011000924],
 'center': [-84.29907462012918, 9.508408455444004, 0]}

In [37]:
latitude = observation_2_tile['center'][1]
longitude = observation_2_tile['center'][0]

# Note that we specify "tiles=None" because in the next step we're going to set a custom tile to serve as our underlying world map.
map_ = folium.Map(location=(latitude, longitude), zoom_start=9, tiles=None, tooltip = 'test tool tip')
# Specify a custom imagery source for the underlying map
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}.png', name='ESRI World Imagery', attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',overlay='True').add_to(map_)
# Add place labels on top
folium.TileLayer(tiles='https://server.arcgisonline.com/arcgis/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}.png',name='ESRI World Boundaries and Places',attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',overlay='True').add_to(map_)

# Use the 'TileLayer' library display the raster layer
map_layer = TileLayer(
    tiles=observation_2_tile["tiles"][0], # Path to retrieve the tile
    name=f'{observation_2.assets[variable.value].title}', # Give this layer a title
    overlay='True', # The layer can be overlaid on the map
    attr="GHG", # Set the attribution
    opacity=0.5, # Adjust the transparency of the layer
)
map_layer.add_to(map_)

# Adjust map elements 
folium.LayerControl(collapsed=False, position='topright').add_to(map_)

# Add colorbar
# We can use one of 'generate_html_colorbar' from the 'ghgc_utils' module 
# to create an HTML colorbar representation.
legend_html = ghgc_utils.generate_html_colorbar(
                color_map,
                rescale_values,
                label=f'{variable.value} ({units.get(variable.value)})'
    )

# Add colorbar to the map
map_.get_root().html.add_child(folium.Element(legend_html))


# Visualizing the map
map_

## Summary

In this notebook, we have successfully explored and visualized the Global Mangrove Distribution, Aboveground Biomass, and Canopy Height dataset. The notebook demonstrated:

1.  **Data Access**: Connected to the US GHG Center STAC API to access the global mangrove dataset
2.  **Data Exploration**: Discovered 121 regional items covering mangrove forests worldwide
3.  **Variable Analysis**: Explored three key mangrove variables:
    - Aboveground biomass (Mg/ha) - critical for carbon storage assessment
    - Maximum canopy height (m) - indicates forest maturity
    - Basal area weighted height (m) - reflects forest structure
4.  **Interactive Visualization**: Created interactive maps showing mangrove distribution in Australia and other regions
5.  **Zonal Statistics**: Calculated detailed statistics for Bathurst Island, Australia, showing mean biomass of ~6.1 Mg/ha
6.  **Global Comparison**: Built interactive widgets to compare mangrove characteristics across different regions

This dataset provides essential information for:
- Coastal ecosystem management and conservation planning
- Understanding global mangrove distribution and health

For questions or feedback about this notebook, please contact us using the [feedback form](https://docs.google.com/forms/d/e/1FAIpQLSeVWCrnca08Gt_qoWYjTo6gnj1BEGL4NCUC9VEiQnXA02gzVQ/viewform).